# Risk Across Tails and Timescales: reproduction notebook
**SAIFA Quant Edge 1.0, Round 1 · Team Gmora**

This notebook reproduces every number, table and figure in `report/QuantEdge_Round1_Report.pdf` and rebuilds the PDF.
Everything runs locally: the price data is bundled in `data/etf_prices.csv`, so nothing is downloaded apart from the
Python packages in the first cell.

**How to run:** open this notebook from the unzipped project folder and choose *Run All*.

| Setting | What happens | Time |
|---|---|---|
| `RUN_FULL = True` (default) | unit tests, full pipeline, PDF rebuild, then the results below | about 10 minutes |
| `RUN_FULL = False` | shows the results already stored in `outputs/` | seconds |

In [ ]:
RUN_FULL = True   # set to False to only display the stored outputs

## 0. Setup
Installs the packages in `requirements.txt` (Python 3.10 to 3.13) into the current kernel.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, sys, json, runpy
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "src" / "run_all.py").exists(), "Open this notebook from the project folder (the one containing src/)."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
print("Python", sys.version.split()[0], "| project folder:", ROOT)

## 1. Unit tests
Checks the statistical building blocks: copula selection recovers the true family, the wavelet decomposition adds back to the series, test statistics and risk formulas match known values.

In [ ]:
if RUN_FULL:
    import pytest
    assert pytest.main(["-q", "tests"]) == 0, "unit tests failed"

## 2. Pipeline
In-sample horizon analysis (1999 to 2019), rolling out-of-sample forecasts (2020 onward), backtests, Monte Carlo seed robustness, figures and the desk recommendation. Results go to `outputs/` and `report/assets/`.

In [ ]:
if RUN_FULL:
    from src.run_all import main as run_pipeline
    run_pipeline()

## 3. Report PDF

In [ ]:
if RUN_FULL:
    runpy.run_path(str(ROOT / "scripts" / "build_report_pdf.py"), run_name="__main__")
print("Report:", ROOT / "report" / "QuantEdge_Round1_Report.pdf")

## 4. Results
### (a) Does tail dependence change with the horizon?

In [ ]:
H = json.loads((ROOT / "outputs" / "results.json").read_text())
cop = pd.read_csv(ROOT / "outputs" / "insample_copula_fits.csv")
display(Markdown("**Full-likelihood copula fits by wavelet band (1999 to 2019)**"))
display(cop[["series", "family", "loglik", "aic", "delta_aic", "aic_winner", "nu", "lambda_l"]])
tests = pd.read_csv(ROOT / "outputs" / "tail_tests_by_band.csv")
display(Markdown("**Bootstrap tests of tail co-movement across bands (q = 5%)**"))
display(tests[tests.q == 0.05][["test", "est", "lo", "hi", "p0"]].rename(columns={"p0": "p_value"}))
display(Image(filename=str(ROOT / "report" / "assets" / "fig1_tail_by_horizon.png")))

In [ ]:
display(Markdown("**Crash co-movement of h-day returns: observed vs implied by daily models (q = 10%)**"))
obs = pd.read_csv(ROOT / "outputs" / "tail_comovement_by_horizon.csv")
obs = obs[(obs.q == 0.10) & (obs["tail"] == "L")].set_index("horizon_days")[["est", "lo", "hi"]]
imp = pd.read_csv(ROOT / "outputs" / "model_implied_tail_by_horizon.csv")
imp = imp[imp.q == 0.10].pivot(index="horizon_days", columns="model", values="chi_L")
display(obs.join(imp))

### (b) What does ignoring it do to measured risk?
**One-day risk, out-of-sample 2020 to 2026**

In [ ]:
rows = []
for m, name in [("HS", "Historical simulation"), ("FHS", "Filtered HS"), ("G", "Gaussian copula"),
                ("T", "Student-t copula"), ("WT", "Wavelet t-copula (H1)")]:
    s = H["oos_1d"][m]
    rows.append({"model": name, "99% breaches": s["var99"]["hits"], "expected": round(s["var99"]["expected_hits"]),
                 "Kupiec p": s["var99"]["p_uc"], "Christoffersen p": s["var99"]["p_ind"], "DQ p": s["var99"]["p_dq"],
                 "Basel red share": s["traffic_light"]["red_share"], "mean ES 97.5%": s["mean_es97_5"],
                 "Z2 p": s["es97_5"]["p_z2"], "FZ0 loss": s["fz0_97_5"]})
display(pd.DataFrame(rows).set_index("model"))
display(Image(filename=str(ROOT / "report" / "assets" / "fig2_oos_var99.png")))
display(Image(filename=str(ROOT / "report" / "assets" / "fig3_traffic_light.png")))

**Ten-day risk: one simplification at a time** (ES gap relative to the full model, the wavelet empirical copula on the H2 band)

In [ ]:
dh = pd.read_csv(ROOT / "outputs" / "oos_10d_forecasts.csv", index_col=0)
rows = []
for m in ["G_sqrt", "G_path", "T_path", "T_join", "WC", "E_join", "WC_emp"]:
    s = H["oos_10d"][m]
    rows.append({"model": m, "mean ES 97.5%": s["mean_es97_5"],
                 "gap vs full model": (dh[f"es97_5_{m}"] / dh["es97_5_WC_emp"] - 1).mean(),
                 "99% breaches": s["var99"]["hits"], "breach loss / ES": s["es97_5"]["realised_over_predicted"],
                 "McNeil-Frey p": s["es97_5"]["mf_p"], "FZ0 loss": s["fz0_97_5"]})
display(pd.DataFrame(rows).set_index("model"))
display(Image(filename=str(ROOT / "report" / "assets" / "fig4_es10d.png")))

**Monte Carlo seed robustness** (main run plus three more seeds)

In [ ]:
rob = pd.read_csv(ROOT / "outputs" / "seed_robustness.csv")
display(rob[["seed"] + [c for c in rob.columns if c.startswith("hits99_")]])

## 5. Desk recommendation

In [ ]:
print((ROOT / "outputs" / "manager_recommendation.txt").read_text())